# 🌲 TASK 8: Ensemble Methods - Random Forest & XGBoost
## Combining Models for Stronger Predictions
### Production-Grade Ensemble Comparison

---

## SETUP: Install & Import Libraries

In [ ]:
# Install XGBoost if not already installed
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])

print('✅ XGBoost installed')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ All libraries imported')

---

## STEP 1: Load & Prepare Data

In [ ]:
from google.colab import files

print('Upload titanic.csv:')
uploaded = files.upload()

df = pd.read_csv('titanic.csv')

print('\n' + '='*80)
print('DATASET LOADED')
print('='*80)
print(f'Shape: {df.shape}')

# Clean data
df_clean = df.copy()

# Feature engineering
df_clean['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df_clean['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
title_mapping = {'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master'}
df_clean['Title'] = df_clean['Title'].map(title_mapping).fillna('Other')
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)

# Drop unnecessary columns
df_clean = df_clean.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# Handle missing values
df_clean['Age'].fillna(df_clean['Age'].median(), inplace=True)
df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0], inplace=True)

# Encode categorical
categorical_cols = ['Sex', 'Embarked', 'Title']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])

X = df_clean.drop('Survived', axis=1)
y = df_clean['Survived']

print(f'\nFeatures: {X.shape[1]}')
print(f'Samples: {X.shape[0]}')
print(f'Churn Rate: {y.sum()/len(y)*100:.1f}%')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain: {len(X_train)}, Test: {len(X_test)}')
print(f'✓ Data ready')

---

## STEP 2: Baseline Models (From Previous Tasks)

In [ ]:
print('\n' + '='*80)
print('BASELINE MODELS')
print('='*80)

# Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
y_proba_lr = lr_model.predict_proba(X_test)[:, 1]

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_proba_lr)

print('\n1. LOGISTIC REGRESSION (Task 3)')
print(f'   Accuracy: {lr_accuracy:.4f}')
print(f'   F1-Score: {lr_f1:.4f}')
print(f'   ROC-AUC: {lr_auc:.4f}')

# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10, class_weight='balanced')
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
y_proba_dt = dt_model.predict_proba(X_test)[:, 1]

dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
dt_recall = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)
dt_auc = roc_auc_score(y_test, y_proba_dt)

print('\n2. DECISION TREE (Task 3)')
print(f'   Accuracy: {dt_accuracy:.4f}')
print(f'   F1-Score: {dt_f1:.4f}')
print(f'   ROC-AUC: {dt_auc:.4f}')

print('\n✅ Baseline models trained')

---

## STEP 3: Train Random Forest Classifier

In [ ]:
print('\n' + '='*80)
print('RANDOM FOREST CLASSIFIER')
print('='*80)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_proba_rf)

print('\n✅ Random Forest trained')
print(f'\nConfiguration:')
print(f'  • n_estimators: 100 trees')
print(f'  • max_depth: 10')
print(f'  • min_samples_split: 10')
print(f'  • Parallel: Yes (n_jobs=-1)')

print(f'\nPerformance:')
print(f'  Accuracy: {rf_accuracy:.4f}')
print(f'  Precision: {rf_precision:.4f}')
print(f'  Recall: {rf_recall:.4f}')
print(f'  F1-Score: {rf_f1:.4f}')
print(f'  ROC-AUC: {rf_auc:.4f}')

---

## STEP 4: Train XGBoost Classifier

In [ ]:
print('\n' + '='*80)
print('XGBOOST CLASSIFIER')
print('='*80)

xgb_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum(),
    eval_metric='logloss',
    verbosity=0
)

xgb_model.fit(X_train, y_train, verbose=0)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb)
xgb_auc = roc_auc_score(y_test, y_proba_xgb)

print('\n✅ XGBoost trained')
print(f'\nConfiguration:')
print(f'  • n_estimators: 100 boosted trees')
print(f'  • max_depth: 6')
print(f'  • learning_rate: 0.1')
print(f'  • subsample: 0.8 (80% of rows per tree)')
print(f'  • colsample_bytree: 0.8 (80% of columns per tree)')

print(f'\nPerformance:')
print(f'  Accuracy: {xgb_accuracy:.4f}')
print(f'  Precision: {xgb_precision:.4f}')
print(f'  Recall: {xgb_recall:.4f}')
print(f'  F1-Score: {xgb_f1:.4f}')
print(f'  ROC-AUC: {xgb_auc:.4f}')

---

## STEP 5: Comprehensive Model Comparison

In [ ]:
print('\n' + '='*80)
print('MODEL COMPARISON TABLE')
print('='*80)

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'Accuracy': [f'{lr_accuracy:.4f}', f'{dt_accuracy:.4f}', f'{rf_accuracy:.4f}', f'{xgb_accuracy:.4f}'],
    'Precision': [f'{lr_precision:.4f}', f'{dt_precision:.4f}', f'{rf_precision:.4f}', f'{xgb_precision:.4f}'],
    'Recall': [f'{lr_recall:.4f}', f'{dt_recall:.4f}', f'{rf_recall:.4f}', f'{xgb_recall:.4f}'],
    'F1-Score': [f'{lr_f1:.4f}', f'{dt_f1:.4f}', f'{rf_f1:.4f}', f'{xgb_f1:.4f}'],
    'ROC-AUC': [f'{lr_auc:.4f}', f'{dt_auc:.4f}', f'{rf_auc:.4f}', f'{xgb_auc:.4f}']
})

print(f'\n{comparison.to_string(index=False)}')

# Find winners
models = ['LR', 'DT', 'RF', 'XGB']
accuracies = [lr_accuracy, dt_accuracy, rf_accuracy, xgb_accuracy]
f1_scores = [lr_f1, dt_f1, rf_f1, xgb_f1]
aucs = [lr_auc, dt_auc, rf_auc, xgb_auc]

print(f'\n' + '-'*80)
print('WINNERS:')
print('-'*80)
print(f'🏆 Best Accuracy: {models[np.argmax(accuracies)]} ({max(accuracies):.4f})')
print(f'🏆 Best F1-Score: {models[np.argmax(f1_scores)]} ({max(f1_scores):.4f})')
print(f'🏆 Best ROC-AUC: {models[np.argmax(aucs)]} ({max(aucs):.4f})')

---

## STEP 6: Feature Importance - Random Forest

In [ ]:
print('\n' + '='*80)
print('FEATURE IMPORTANCE - RANDOM FOREST')
print('='*80)

rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('\nTop 10 Most Important Features:')
for i, row in rf_importance.head(10).iterrows():
    print(f'{row["Feature"]:20s}: {row["Importance"]*100:6.2f}%')

print('\n🎯 TOP 3 FEATURES:')
for idx, (i, row) in enumerate(rf_importance.head(3).iterrows(), 1):
    print(f'{idx}. {row["Feature"]:20s} ({row["Importance"]*100:.2f}%)')

---

## STEP 7: Feature Importance - XGBoost

In [ ]:
print('\n' + '='*80)
print('FEATURE IMPORTANCE - XGBOOST')
print('='*80)

xgb_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('\nTop 10 Most Important Features:')
for i, row in xgb_importance.head(10).iterrows():
    print(f'{row["Feature"]:20s}: {row["Importance"]*100:6.2f}%')

print('\n🎯 TOP 3 FEATURES:')
for idx, (i, row) in enumerate(xgb_importance.head(3).iterrows(), 1):
    print(f'{idx}. {row["Feature"]:20s} ({row["Importance"]*100:.2f}%)')

---

## STEP 8: Feature Importance Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Feature Importance Comparison - Random Forest vs XGBoost', fontsize=14, fontweight='bold')

# Random Forest
top10_rf = rf_importance.head(10)
axes[0].barh(range(len(top10_rf)), top10_rf['Importance'].values, color='#3498db', alpha=0.8, edgecolor='black')
axes[0].set_yticks(range(len(top10_rf)))
axes[0].set_yticklabels(top10_rf['Feature'].values)
axes[0].set_xlabel('Importance Score')
axes[0].set_title('Random Forest - Top 10 Features', fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3, axis='x')

# XGBoost
top10_xgb = xgb_importance.head(10)
axes[1].barh(range(len(top10_xgb)), top10_xgb['Importance'].values, color='#e74c3c', alpha=0.8, edgecolor='black')
axes[1].set_yticks(range(len(top10_xgb)))
axes[1].set_yticklabels(top10_xgb['Feature'].values)
axes[1].set_xlabel('Importance Score')
axes[1].set_title('XGBoost - Top 10 Features', fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print('✅ Feature importance comparison complete')

---

## STEP 9: Model Performance Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

# Accuracy, F1, ROC-AUC comparison
models = ['Logistic\nRegression', 'Decision\nTree', 'Random\nForest', 'XGBoost']
accuracies = [lr_accuracy, dt_accuracy, rf_accuracy, xgb_accuracy]
f1_scores = [lr_f1, dt_f1, rf_f1, xgb_f1]
aucs = [lr_auc, dt_auc, rf_auc, xgb_auc]

x = np.arange(len(models))
width = 0.25

axes[0].bar(x - width, accuracies, width, label='Accuracy', color='#3498db', alpha=0.8, edgecolor='black')
axes[0].bar(x, f1_scores, width, label='F1-Score', color='#2ecc71', alpha=0.8, edgecolor='black')
axes[0].bar(x + width, aucs, width, label='ROC-AUC', color='#e74c3c', alpha=0.8, edgecolor='black')
axes[0].set_ylabel('Score')
axes[0].set_title('Overall Performance Metrics', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].legend()
axes[0].set_ylim([0.75, 1.0])
axes[0].grid(True, alpha=0.3, axis='y')

# Improvement over baseline (Logistic Regression)
improvements = [0, (dt_f1-lr_f1)/lr_f1*100, (rf_f1-lr_f1)/lr_f1*100, (xgb_f1-lr_f1)/lr_f1*100]
colors = ['gray', '#ff6b6b' if improvements[1] < 0 else '#51cf66', '#51cf66', '#51cf66']
axes[1].bar(models, improvements, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[1].set_ylabel('Improvement (%)')
axes[1].set_title('F1-Score Improvement vs Baseline (LR)', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for i, v in enumerate(improvements):
    axes[1].text(i, v + (1 if v > 0 else -3), f'{v:+.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('✅ Performance comparison visualization complete')

---

## STEP 10: Random Forest vs XGBoost Explanation

In [ ]:
print('\n' + '='*80)
print('RANDOM FOREST VS XGBOOST: HOW THEY DIFFER')
print('='*80)

explanation = '''
🌲 RANDOM FOREST
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

How it works:
1. Create 100 independent decision trees
2. Each tree trained on random subset of data (bootstrap)
3. Each tree trained on random subset of features
4. Each tree is fully grown (no depth limit)
5. Predictions: Majority vote (classification) or average (regression)

Key Features:
• Parallel: All trees independent, no information shared
• Fast Training: Can train trees in parallel (n_jobs=-1)
• High Variance: Each tree trained differently
• Averaging: Reduces variance by averaging many different trees
• Less prone to overfitting: Diversity helps

Pros:
✓ Fast training
✓ Parallel processing
✓ Good for medium datasets
✓ Robust to outliers
✓ Easy to parallelize
✓ Good baseline model

Cons:
✗ Doesn't learn from mistakes (sequential trees all independent)
✗ Can overfit on noisy data
✗ Large models in memory

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🚀 XGBOOST (Extreme Gradient Boosting)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

How it works:
1. Train tree 1, predict, calculate errors
2. Train tree 2 to correct tree 1's errors
3. Train tree 3 to correct combined errors of trees 1+2
4. Repeat 100 times, each tree learns from previous mistakes
5. Predictions: Sequential sum (each tree builds on last)

Key Features:
• Sequential: Each tree learns from previous tree's mistakes
• Gradient Descent: Minimizes loss function explicitly
• Regularization: Built-in L1/L2 penalty to prevent overfitting
• Shrinkage: Each tree contributes scaled amount (learning_rate)
• Smart Splits: Considers gain from splits
• Handles Imbalance: scale_pos_weight for imbalanced data

Pros:
✓ Often highest accuracy
✓ Learns from mistakes (sequential)
✓ Handles imbalanced data well
✓ Automatic regularization
✓ Faster inference
✓ Winning model in competitions
✓ Better for large datasets

Cons:
✗ Slower training (sequential, can't parallelize)
✗ More hyperparameters to tune
✗ More complex to understand
✗ Prone to overfitting if not regularized
✗ Requires careful tuning

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

QUICK COMPARISON
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

│ Aspect        │ Random Forest │ XGBoost      │
├───────────────┼───────────────┼──────────────┤
│ Build Method  │ Parallel      │ Sequential   │
│ Error Fixing  │ No            │ Yes (gradient)│
│ Learning      │ Averaging     │ Boosting     │
│ Accuracy      │ Good          │ Often Better │
│ Training Speed│ Fast          │ Slower       │
│ Tuning Ease   │ Easy          │ Complex      │
│ Competition   │ Second place  │ First place  │
│ Industry Use  │ Baseline      │ Production   │

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SIMPLE ANALOGY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Random Forest:
  You hire 100 smart people who work independently.
  Each makes decisions separately.
  You take majority vote.
  → Diverse opinions prevent groupthink
  → Strong but doesn't learn from mistakes

XGBoost:
  You hire 1 person for 100 iterations.
  Each iteration, they learn from their previous mistakes.
  Each new attempt fixes old errors.
  → Focused learning
  → Gets better with each iteration
  → Can overfit if not careful

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
'''

print(explanation)

---

## STEP 11: Confusion Matrices - Ensemble Models

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Confusion Matrices - All Models', fontsize=14, fontweight='bold')

models_cm = [
    (y_pred_lr, 'Logistic Regression', axes[0, 0]),
    (y_pred_dt, 'Decision Tree', axes[0, 1]),
    (y_pred_rf, 'Random Forest', axes[1, 0]),
    (y_pred_xgb, 'XGBoost', axes[1, 1])
]

for y_pred, title, ax in models_cm:
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Not Survived', 'Survived'],
                yticklabels=['Not Survived', 'Survived'])
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

print('✅ Confusion matrices displayed')

---

## STEP 12: Final Summary

In [ ]:
print('\n' + '='*80)
print('🎉 TASK 8 COMPLETE - ENSEMBLE METHODS')
print('='*80)

print(f'\n📊 DATASET')
print(f'Total Samples: {len(df)}')
print(f'Features: {X.shape[1]}')
print(f'Train/Test: {len(X_train)}/{len(X_test)}')

print(f'\n🏆 MODEL RANKINGS (by F1-Score):')
models_ranked = [
    ('XGBoost', xgb_f1),
    ('Random Forest', rf_f1),
    ('Logistic Regression', lr_f1),
    ('Decision Tree', dt_f1)
]
models_ranked_sorted = sorted(models_ranked, key=lambda x: x[1], reverse=True)
for i, (model, score) in enumerate(models_ranked_sorted, 1):
    print(f'{i}. {model:25s}: {score:.4f}')

print(f'\n✅ KEY INSIGHTS')
print(f'✓ Ensemble methods (RF, XGB) outperform single models')
print(f'✓ XGBoost typically achieves highest accuracy')
print(f'✓ Random Forest faster training, easier tuning')
print(f'✓ Different models highlight different features')
print(f'✓ Feature importance varies by algorithm')

print(f'\n✅ TASK 8 COMPLETE!')
print('='*80)